# NASA EPIC Image with OpenCV

This notebook loads EPIC metadata from `app.routes.nasa`, builds the NASA image URL, downloads the image, and converts it for OpenCV.

In [ ]:
import importlib
import os
import sys
from io import BytesIO
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

cwd = Path.cwd().resolve()
backend_dir = next(
    path for path in [cwd, *cwd.parents]
    if (path / "app" / "routes" / "nasa.py").exists()
)

if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from app.routes import nasa
importlib.reload(nasa)

print(nasa.__file__)

In [ ]:
nasa_data = nasa.get_nasa_data()

print(type(nasa_data), len(nasa_data))
nasa_data[0]

In [ ]:
def epic_image_url(item, ext="png"):
    date = item["date"].split()[0]
    year, month, day = date.split("-")
    image_name = item["image"]

    return (
        f"https://api.nasa.gov/EPIC/archive/natural/"
        f"{year}/{month}/{day}/{ext}/{image_name}.{ext}"
        f"?api_key={nasa.API_KEY}"
    )

image_urls = [epic_image_url(item) for item in nasa_data]
image_urls[:3]

In [ ]:
image_url = image_urls[0]
response = requests.get(image_url, timeout=20)
response.raise_for_status()

pil_image = Image.open(BytesIO(response.content)).convert("RGB")
image_rgb = np.array(pil_image, dtype=np.uint8)
image_rgb = np.ascontiguousarray(image_rgb)

print(type(image_rgb), image_rgb.shape, image_rgb.dtype)
pil_image

In [ ]:
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 80, 160)

print(image_bgr.shape, gray.shape, edges.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("RGB")
axes[0].axis("off")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[1].axis("off")

axes[2].imshow(edges, cmap="gray")
axes[2].set_title("Canny edges")
axes[2].axis("off")

plt.tight_layout();